# 🎮 MUKEUS VIDEO ENHANCER — Free Google Colab GPU Processing

Run **MUKEUS VIDEO ENHANCER** virtually on **Google Colab's Free NVIDIA T4 GPU (16 GB VRAM)** with zero load on your local PC!

### Instructions:
1. In Colab top menu, click **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ **Save**.
2. Run **Cell 1** to verify GPU & install dependencies.
3. Run **Cell 2** to clone repository and launch your live web app!

In [ ]:
# CELL 1: Check GPU & Install Dependencies
!nvidia-smi
import torch
print('='*50)
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('✅ GPU ACTIVE:', torch.cuda.get_device_name(0))
else:
    print('⚠️ CRITICAL WARNING: Running on CPU mode!')
    print('👉 Please click top menu: Runtime ➔ Change runtime type ➔ Select T4 GPU ➔ Save')
print('='*50)
!apt-get update -qq && apt-get install -y -qq ffmpeg npm
!pip install -q fastapi "uvicorn[standard]" python-multipart pydantic torch torchvision opencv-python numpy imageio-ffmpeg pyngrok nest_asyncio pycloudflared
!npm install -g localtunnel

In [ ]:
# CELL 2: Clone Git Repo & Launch Web App with Multiple Tunnel Fallbacks
import os, sys, time, subprocess

REPO_URL = "https://github.com/mukesh-ram/Mukeus_AI_Video_Enhancer.git"
APP_DIR = "/content/Mukeus_AI_Video_Enhancer"

if not os.path.exists(APP_DIR):
    print(f"📥 Cloning MUKEUS repository from {REPO_URL}...")
    !git clone {REPO_URL} {APP_DIR}
else:
    print("🔄 Pulling latest changes...")
    !git -C {APP_DIR} pull

if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
os.chdir(APP_DIR)

# 1. Start FastAPI server in dedicated process
print("⚡ Starting FastAPI Uvicorn Server on port 8000...")
server_process = subprocess.Popen([
    sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"
])
time.sleep(3)

# 2. Official Google Colab Built-in Proxy Link
try:
    from google.colab.output import eval_js
    colab_proxy_url = eval_js('google.colab.kernel.proxyPort(8000)')
    print("\n" + "="*65)
    print("🌐 OFFICIAL COLAB DIRECT LINK (100% Reliable):")
    print(f"👉 {colab_proxy_url}")
    print("="*65)
except Exception as e:
    pass

# 3. Cloudflare Tunnel Link
try:
    from pycloudflared import try_cloudflare
    cf_tunnel = try_cloudflare(port=8000)
    print("\n⚡ CLOUDFLARE LINK:")
    print(f"👉 {cf_tunnel.tunnel}")
    print("="*65 + "\n")
except Exception as e:
    pass

# 4. Get Public IP for LocalTunnel password if needed
!curl -s ipv4.icanhazip.com > /tmp/ip.txt
try:
    with open('/tmp/ip.txt') as f:
        pub_ip = f.read().strip()
    print(f"💡 LocalTunnel Password (if prompted): {pub_ip}")
except:
    pass

# Launch localtunnel in background
subprocess.Popen(["npx", "localtunnel", "--port", "8000"])

try:
    server_process.wait()
except KeyboardInterrupt:
    server_process.terminate()